# 참고용 사본 (Reference copy)

이 노트북은 원 연구 노트북에서 출력, 실행 횟수, 로컬 경로, 키 값을 제거한 **참고용 사본**이다. 뉴스 원문, 일별·분봉 가격, 경매 거래대금 등 **공개하지 않은 데이터**(`news_tokenized_labeled*.pkl`, `daily_adjusted_close.parquet` 등)가 있어야 처음부터 끝까지 실행되며, 이 저장소만으로는 실행되지 않는다. 분석 구조와 코드를 읽는 용도다. 이 저장소에서 바로 실행되는 것은 `00_public_sample_workflow.ipynb`와 `../code/`뿐이다.

This is a reference copy of an original research notebook: outputs, execution counts, local paths and keys were removed. It needs non-public data (news text, daily/minute prices, auction values) and will not run from this repository alone.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("intraday_experiment").resolve()))
from intraday_event_tools import (
    prepare_first_news_quintile_events,
    query_quintile_event_path_summary,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:,.6f}".format)


---
### **NB Sentiment 5분위별 장중 가격 경로**


In [ ]:
# =============================================================================
# 1. 파라미터
# =============================================================================
STOCK_DIR = Path("../data/stock")

N_HOURS_BEFORE = 2
SCORE_COL = "NB_sentiment"

EVENT_PATH = Path("intraday_first_news_nb_quintile_events.parquet")
PATH_SUMMARY_PATH = Path("intraday_first_news_nb_quintile_path_2h_to_close.parquet")
NEXT_OPEN_PATH = Path("intraday_first_news_nb_quintile_next_open.parquet")

FORCE_RECOMPUTE = False


In [ ]:
# =============================================================================
# 2. 데이터 불러오기 또는 재계산
# =============================================================================
if (
    EVENT_PATH.exists()
    and PATH_SUMMARY_PATH.exists()
    and NEXT_OPEN_PATH.exists()
    and not FORCE_RECOMPUTE
):
    events_with_base = pd.read_parquet(EVENT_PATH)
    path_summary = pd.read_parquet(PATH_SUMMARY_PATH)
    next_open_summary = pd.read_parquet(NEXT_OPEN_PATH)
else:
    df = pd.read_pickle("news_tokenized_labeled_scored.pkl")
    price_wide = pd.read_parquet("price_wide.parquet")

    events = prepare_first_news_quintile_events(
        df,
        price_wide,
        score_col=SCORE_COL,
        hours_before=N_HOURS_BEFORE,
    )

    events_with_base, path_summary, next_open_summary = query_quintile_event_path_summary(
        events,
        STOCK_DIR,
    )

    events_with_base.to_parquet(EVENT_PATH, index=False)
    path_summary.to_parquet(PATH_SUMMARY_PATH, index=False)
    next_open_summary.to_parquet(NEXT_OPEN_PATH, index=False)

print(events_with_base.shape)
print(path_summary.shape)
print(next_open_summary.shape)


In [ ]:
# =============================================================================
# 3. 표본 확인
# =============================================================================
event_summary = (
    events_with_base
    .groupby("score_quintile")
    .agg(
        events=("event_id", "nunique"),
        dates=("date", "nunique"),
        avg_NB=("NB_sentiment", "mean"),
        min_NB=("NB_sentiment", "min"),
        max_NB=("NB_sentiment", "max"),
        next_open_valid=("next_open_ret", "count"),
        next_open_mean=("next_open_ret", "mean"),
        next_open_median=("next_open_ret", "median"),
    )
)

event_summary


In [ ]:
next_open_summary


---
### **5분위별 가격 경로 그래프**

`0분`은 뉴스 발간 후 첫 기준가격이 잡힌 시점입니다.  
오른쪽의 `next open` 점은 익일 09:00~09:05 VWAP 수익률이며, 시각 간격을 정확한 분 단위로 표시한 것이 아니라 비교를 위해 장중 경로 오른쪽에 따로 배치했습니다.


In [ ]:
# =============================================================================
# 4. 5개 그래프 출력
# =============================================================================
terminal_x = path_summary["offset_min"].max() + 30
colors = {
    1: "#4c78a8",
    2: "#72b7b2",
    3: "#f2cf5b",
    4: "#f58518",
    5: "#e45756",
}

fig, axes = plt.subplots(5, 1, figsize=(12, 18), sharex=True)

for q, ax in zip(range(1, 6), axes):
    sub = path_summary[path_summary["score_quintile"].eq(q)].sort_values("offset_min")
    terminal = next_open_summary[next_open_summary["score_quintile"].eq(q)]

    ax.plot(
        sub["offset_min"],
        sub["mean_ret"] * 10000,
        color=colors[q],
        linewidth=1.8,
        label=f"Q{q} intraday path",
    )

    if not terminal.empty:
        y = terminal["mean_ret"].iloc[0] * 10000
        ax.scatter(
            [terminal_x],
            [y],
            color=colors[q],
            edgecolor="black",
            linewidth=0.7,
            s=80,
            zorder=4,
            label="next open",
        )
        ax.annotate(
            f"{y:,.1f} bp",
            xy=(terminal_x, y),
            xytext=(8, 0),
            textcoords="offset points",
            va="center",
        )

    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.7)
    ax.set_title(f"NB sentiment daily quintile Q{q}")
    ax.set_ylabel("Return (bp)")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left")

axes[-1].set_xlabel(f"Minutes from news baseline; left side starts {N_HOURS_BEFORE} hours before news")
axes[-1].set_xticks(
    sorted(set(list(range(-N_HOURS_BEFORE * 60, int(path_summary["offset_min"].max()) + 1, 60)) + [0, terminal_x]))
)
axes[-1].set_xticklabels(
    [
        "next open" if x == terminal_x else str(int(x))
        for x in axes[-1].get_xticks()
    ],
    rotation=0,
)

plt.tight_layout()
plt.show()


---
### **각 offset의 관측 이벤트 수**


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for q in range(1, 6):
    sub = path_summary[path_summary["score_quintile"].eq(q)].sort_values("offset_min")
    ax.plot(sub["offset_min"], sub["n_events"], color=colors[q], linewidth=1.4, label=f"Q{q}")

ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.7)
ax.set_xlabel("Minutes from news baseline")
ax.set_ylabel("Number of events")
ax.set_title("Available Events by Offset")
ax.legend(ncol=5)
ax.grid(alpha=0.25)
plt.show()


---
### **참고: 이벤트 예시**


In [ ]:
events_with_base[[
    "date", "gicode", "company_name", "published_ts", "score_quintile",
    "NB_sentiment", "base_bar", "base_price", "next_open_vwap", "next_open_ret",
]].head(20)
